# SE-RISK-TRADE: Sistema Especialista Híbrido de Gestão de Risco Operacional na Bolsa de Valores
## Avaliação A1 — Inteligência Artificial e LFA
**Professora:** Profa. Kadidja Valéria  
**Integrantes:** João Paulo de Oliveira Paiva e Danilo Hellu  
**Instituição:** UDF Centro Universitário — Novembro de 2026  
**Repositório GitHub:** https://github.com/Danilove1106/InteligenciaArtificial_LinguagensFormais_Automatos

---
### Objetivo do Trabalho
Desenvolver um Sistema Especialista Híbrido que combina **Aprendizado de Máquina Supervisionado (Árvore de Decisão)** e um **Motor de Inferência Simbólico Baseado em Regras (Encadeamento para Frente)** para a governança de risco e execução de ordens na Bolsa de Valores (B3 / mercado financeiro), em conformidade com as diretrizes da **CVM (Resolução nº 35/2021)** e do **Banco Central do Brasil (Bacen / dados.gov.br)**.

## 1. Preparação do Ambiente
Importação das bibliotecas essenciais para análise estatística, aprendizado de máquina e visualização gráfica.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print("Bibliotecas importadas com sucesso.")

## 2. Dataset Rotulado de Séries de Mercado
O dataset é composto por métricas técnicas e operacionais de negociação:
- : Volatilidade média calculada pelo Average True Range (normalizada).
- : Força da tendência pelo Average Directional Index (0 a 100).
- : Custo de liquidez / spread médio de oferta na B3.
- : Variação percentual recente do ativo.
- : Rótulo de classificação ('favoravel' vs 'alto_risco').

In [ ]:
dados = pd.read_csv('dataset_mercado_financeiro.csv')
print("=== Primeiras Linhas do Dataset ===")
print(dados.head(10).to_string(index=False))

print("
Distribuição das Classes no Dataset:")
print(dados['regime_mercado'].value_counts())

## 3. Separação de Treinamento e Teste
Separamos 75% dos dados para treinamento e 25% para avaliação em conjunto de teste independente. O parâmetro  garante a paridade de proporções de cada classe e  garante reprodutibilidade estrita.

In [ ]:
X = dados[['volatilidade_atr', 'adx_tendencia', 'spread_pontos', 'retorno_recente_pct']]
y = dados['regime_mercado']

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Total de Amostras: {len(dados)} | Treino: {len(X_treino)} | Teste: {len(X_teste)}")

## 4. Treinamento da Árvore de Decisão e Avaliação
Ajustamos o classificador  limitando a profundidade () para manter máxima interpretabilidade e prevenir overfitting.

In [ ]:
modelo_arvore = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo_arvore.fit(X_treino, y_treino)

y_pred = modelo_arvore.predict(X_teste)
acuracia = accuracy_score(y_teste, y_pred)
matriz = confusion_matrix(y_teste, y_pred, labels=['favoravel', 'alto_risco'])

print(f"Acurácia no Conjunto de Teste: {acuracia * 100:.2f}%")
print("
Matriz de Confusão:")
print(matriz)

disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=['favoravel', 'alto_risco'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão - Classificação de Regime de Risco")
plt.show()

## 5. Visualização das Regras da Árvore de Decisão
A visualização abaixo demonstra a extração das regras estatísticas aprendidas pelo algoritmo nos dados de treinamento.

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(
    modelo_arvore,
    feature_names=X.columns,
    class_names=modelo_arvore.classes_,
    filled=True,
    rounded=True
)
plt.title("Árvore de Decisão: Classificação de Regime de Risco Operacional")
plt.show()

print("
Regras em formato textual extraído:")
print(export_text(modelo_arvore, feature_names=list(X.columns)))

## 6. Base de Conhecimento e Motor de Inferência (Lógica Simbólica)
A base de conhecimento formaliza as regras SE... ENTÃO... regulatórias da CVM e da B3, garantindo governança com encadeamento para frente e tratamento de incerteza ('?').

In [ ]:
from regras import REGRAS_RISCO_TRADE
from motor_inferencia import MotorInferenciaRisco

motor = MotorInferenciaRisco()
print(f"Motor de Inferência inicializado com {len(motor.regras)} regras regulatórias cadastradas.")

## 7. Integração Híbrida: Machine Learning + Sistema Especialista
A função  recebe tanto métricas numéricas quanto fatos regulatórios, obtém a predição da árvore e aciona o motor de inferência.

In [ ]:
from main import SistemaEspecialistaTrade, imprimir_relatorio_caso

sistema_completo = SistemaEspecialistaTrade()

# Demonstração de caso prático: Condições Ideais de Mercado
caso_exemplo = {
    "descricao": "Condições Ideais de Liquidez e Alinhamento Técnico Padrão",
    "atr": 15.0, "adx": 31.0, "spread": 18, "retorno": 1.10,
    "cb": "não", "dd": "não", "macro": "não", "perdas": "não",
    "spread_ok": "sim", "adx_forte": "sim", "horario": "sim"
}

resultado = sistema_completo.avaliar_operacao(
    volatilidade_atr=caso_exemplo["atr"],
    adx_tendencia=caso_exemplo["adx"],
    spread_pontos=caso_exemplo["spread"],
    retorno_recente_pct=caso_exemplo["retorno"],
    circuit_breaker_bolsa=caso_exemplo["cb"],
    drawdown_diario_excedido=caso_exemplo["dd"],
    evento_macro_proximo=caso_exemplo["macro"],
    perdas_consecutivas_mesma_direcao=caso_exemplo["perdas"],
    spread_dentro_do_limite=caso_exemplo["spread_ok"],
    adx_tendencia_forte=caso_exemplo["adx_forte"],
    horario_operacional_permitido=caso_exemplo["horario"]
)

imprimir_relatorio_caso(1, caso_exemplo, resultado)

## 8. Consulta Interativa
Modifique os parâmetros abaixo para simular diferentes cenários de mercado e auditar as decisões tomadas pelo Sistema Especialista.

In [ ]:
# Parâmetros de Entrada da Simulação
atr_input = 52.0
adx_input = 13.0
spread_input = 80
retorno_input = -4.2
circuit_breaker_input = "não"
drawdown_input = "sim" # Simulando teto de perda atingido

res_simulacao = sistema_completo.avaliar_operacao(
    volatilidade_atr=atr_input,
    adx_tendencia=adx_input,
    spread_pontos=spread_input,
    retorno_recente_pct=retorno_input,
    circuit_breaker_bolsa=circuit_breaker_input,
    drawdown_diario_excedido=drawdown_input
)

print("=== RESULTADO DA SIMULAÇÃO INTERATIVA ===")
print(f"Predição da Árvore de Decisão: {res_simulacao['regime_mercado_ml']}")
print(f"Regra Disparada: [{res_simulacao['regra_id']}] {res_simulacao['regra_nome']}")
print(f"Decisão Recomendada: {res_simulacao['acao_recomendada']}")
print(f"Justificativa: {res_simulacao['explicacao']}")
print(f"Fundamento Legal: {res_simulacao['fundamento_legal']}")
print(f"Trilha Explicativa: {res_simulacao['explicacao_como']}")

## 9. Conclusões e Referências

### Conclusões Técnicas
1. **Complementaridade Híbrida:** A inteligência estatística da árvore de decisão fornece sensibilidade aos padrões de volatilidade e liquidez, enquanto a lógica simbólica garante que nenhuma ordem viole salvaguardas mandatórias de risco (CVM/B3).
2. **Explicabilidade e Auditabilidade:** A preservação do rastro de inferência atende às diretrizes de XAI (*Explainable AI*), permitindo a auditoria em tempo constante de cada decisão tomada.
3. **Tratamento de Incerteza:** O tratamento do estado desconhecido ('?') assegura que falhas de telemetria resultem em paralisação preventiva, evitando acidentes operacionais.

### Referências
- BRASIL. Comissão de Valores Mobiliários (CVM). **Resolução CVM nº 35, de 5 de março de 2021** (Regulamenta a atividade de negociação automatizada e sistemas de negociação algorítmica).
- BANCO CENTRAL DO BRASIL (BACEN). **Sistema Gerenciador de Séries Temporais (SGS)**. Disponível em: https://dadosabertos.bcb.gov.br.
- B3 S.A. – BRASIL, BOLSA, BALCÃO. **Manual de Procedimentos Operacionais de Negociação**.
- RUSSELL, Stuart; NORVIG, Peter. **Inteligência Artificial: Uma Abordagem Moderna**. 3. ed. Rio de Janeiro: Elsevier, 2013.
- LACERDA JÚNIOR, Francisco Pires de. **Uso ético da inteligência artificial generativa na educação superior — uma revisão integrativa (2023–2026)**. Revista Foco, v. 19, n. 8, 2026.